# Индивидуальная работа №2
## Продвинутый статистический анализ данных и методы машинного обучения

**Тема:** Rail Equipment Accident/Incident Data (Form 54) — Unique Train Accidents (Not at Grade Crossings)

**Студенты:** Doicov Pavel, Iachimenko Alexandr, Morozan Nikita  
**Группа:** IA2303  
**Источник данных:** U.S. Department of Transportation — Federal Railroad Administration (FRA)  
`https://data.transportation.gov/Railroads/`


---
# 1. Введение и обоснование темы

## 1.1. Постановка задачи

В данной работе исследуется датасет **Rail Equipment Accident/Incident Data (Form 54)** — официальная база данных США об авариях на железнодорожном транспорте.

Каждый инцидент содержит информацию о:
- типе аварии (сход с рельсов, столкновение и др.)
- скорости и характеристиках поезда
- состоянии путей и погоде
- материальном ущербе и пострадавших

## 1.2. Цели исследования

1. Провести очистку и предобработку данных
2. Выполнить разведочный анализ (EDA) с 20+ визуализациями
3. Построить модели **линейной, логистической и мультиномиальной регрессии**
4. Применить **PCA** для снижения размерности
5. Провести **анализ временных рядов (ARIMA)**
6. Интерпретировать результаты для повышения безопасности перевозок

## 1.3. Описание датасета

| Параметр | Значение |
|----------|----------|
| Источник | U.S. Federal Railroad Administration (FRA) |
| Объём | ~181 737 наблюдений |
| Признаки | 155 колонок |
| Числовые | ~75 признаков |
| Категориальные | ~80 признаков |
| Обновление | Март 2026 |


---
# 2. Предобработка и очистка данных

## 2.1. Загрузка библиотек и данных


In [260]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    PLOTLY_AVAILABLE = True
    pio.renderers.default = 'plotly_mimetype+notebook'
except ImportError:
    go = None
    pio = None
    PLOTLY_AVAILABLE = False

from scipy import stats
from scipy.stats import zscore, shapiro, pearsonr
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, balanced_accuracy_score
)
from sklearn.decomposition import PCA
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
from itertools import combinations

set_matplotlib_formats('retina')

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#2563EB', '#DC2626', '#16A34A', '#D97706', '#7C3AED', '#0891B2']
sns.set_palette(PALETTE)
np.random.seed(42)

MODEL_NUMERIC_FEATURES = [
    'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'Temperature',
    'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars',
    'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars',
    'Derailed Head End Locomotives', 'Derailed Mid Train Manual Locomotives',
    'Derailed Mid Train Remote Locomotives', 'Derailed Rear End Manual Locomotives',
    'Derailed Rear End Remote Locomotives', 'Derailed Cabooses',
    'Track Class', 'Track Density'
]
MODEL_CATEGORICAL_FEATURES = [
    'Accident Type', 'Weather Condition', 'Visibility', 'Track Type',
    'Train Direction', 'Equipment Type', 'Signalization',
    'Method Of Operation', 'Class'
]


def enrich_operational_features(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    numeric_cols = [c for c in MODEL_NUMERIC_FEATURES + ['Total Damage Cost'] if c in data.columns]
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    if {'Loaded Freight Cars', 'Empty Freight Cars'}.issubset(data.columns):
        data['Total Freight Cars'] = data[['Loaded Freight Cars', 'Empty Freight Cars']].sum(axis=1, min_count=1)

    derailed_cols = [
        c for c in [
            'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars',
            'Derailed Head End Locomotives', 'Derailed Mid Train Manual Locomotives',
            'Derailed Mid Train Remote Locomotives', 'Derailed Rear End Manual Locomotives',
            'Derailed Rear End Remote Locomotives', 'Derailed Cabooses'
        ] if c in data.columns
    ]
    if derailed_cols:
        data['Total Derailed Units'] = data[derailed_cols].sum(axis=1, min_count=1)

    if {'Train Speed', 'Maximum Speed'}.issubset(data.columns):
        data['Speed Utilization'] = data['Train Speed'] / data['Maximum Speed'].replace({0: np.nan})

    if {'Total Derailed Units', 'Total Freight Cars'}.issubset(data.columns):
        data['Derailed Share'] = data['Total Derailed Units'] / data['Total Freight Cars'].replace({0: np.nan})

    if 'Gross Tonnage' in data.columns:
        data['log_Gross Tonnage'] = np.log1p(data['Gross Tonnage'])

    return data


def make_preprocessor(num_cols, cat_cols, scale_numeric=True):
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))

    return ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False))
        ]), cat_cols),
    ])


def pretty_feature_name(name: str) -> str:
    clean = name.replace('num__', '').replace('cat__', '')
    clean = clean.replace('_', ' = ')
    return clean


def show_current_matplotlib_figure():
    fig = plt.gcf()
    display(fig)
    plt.close(fig)


print('Библиотеки и helper-функции загружены успешно')
print(f'Интерактивный Plotly доступен: {PLOTLY_AVAILABLE}')


Библиотеки и helper-функции загружены успешно
Интерактивный Plotly доступен: False


In [261]:
#  Загрузка данных 
# Скачать: https://data.transportation.gov/Railroads/
# (Rail Equipment Accident/Incident Data — Unique Train Accidents)

df = pd.read_csv("data.csv", low_memory=False)

print(f" Размер датасета: {df.shape[0]:,} строк × {df.shape[1]} колонок")
print(f"\nПервые 3 строки:")
df.head(3)


,Reporting Railroad Code,Reporting Railroad Name,Year,Accident Number,PDF Link,Accident Year,Accident Month,Other Railroad Code,Other Railroad Name,Other Accident Number,...,Reporting Parent Railroad Name,Reporting Railroad Holding Company,Location,Reporting Railroad Individual Class,Reporting Railroad Passenger,Reporting Railroad Commuter,Reporting Railroad Switching Terminal,Reporting Railroad Tourist,Reporting Railroad Freight,Reporting Railroad Short Line
0,ALS,Alton & Southern Railway,1978,0707,https://safetydata.fra.dot.gov/Officeofsafety/...,78,7,NaN,NaN,NaN,...,Union Pacific Railroad Company,Union Pacific Railroad Company,NaN,Class III,Unassigned,Unassigned,Unassigned,Unassigned,Yes,Yes
1,BN,Burlington Northern Railroad Company,1976,MN199,https://safetydata.fra.dot.gov/Officeofsafety/...,76,1,NaN,NaN,NaN,...,BNSF Railway Company,BNSF Railway Company,NaN,Not Assigned,Not Assigned,Not Assigned,Not Assigned,Not Assigned,Not Assigned,Not Assigned
2,BN,Burlington Northern Railroad Company,1978,PA2045,https://safetydata.fra.dot.gov/Officeofsafety/...,78,12,NaN,NaN,NaN,...,BNSF Railway Company,BNSF Railway Company,NaN,Not Assigned,Not Assigned,Not Assigned,Not Assigned,Not Assigned,Not Assigned,Not Assigned


## 2.2. Обработка пропущенных значений

**Стратегия:**
- Признаки с долей пропусков **> 80%** → удаляются (нерелевантны)
- Числовые признаки → заполняются **медианой** (устойчива к выбросам)
- Категориальные признаки → заполняются строкой **"Unknown"**


In [262]:
#  Анализ пропусков 
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print("Топ-15 признаков по доле пропусков:")
print(missing.head(15).to_string())

#  Удаление признаков с >80% пропусков 
high_missing = missing[missing['missing_pct'] > 80].index.tolist()
df.drop(columns=high_missing, inplace=True)
print(f"\n  Удалено {len(high_missing)} признаков с >80% пропусков")

#  Заполнение пропусков 
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna('Unknown', inplace=True)

print(f" Оставшиеся пропуски: {df.isna().sum().sum()}")
print(f" Итоговый размер: {df.shape}")


 Оставшиеся пропуски: 1606357
 Итоговый размер: (181737, 132)


## 2.3. Выявление и обработка выбросов

Используем два метода:
- **IQR** (межквартильный размах): выброс если значение < Q1 − 1.5·IQR или > Q3 + 1.5·IQR
- **Z-score**: выброс если |z| > 3

**Решение:** выбросы **не удаляются**, т.к. крупные аварии с большим ущербом — реальные события, важные для анализа риска. Для моделей применяется логарифмирование.


In [263]:
key_numeric = [c for c in ['Train Speed','Maximum Speed','Gross Tonnage',
                           'Equipment Damage Cost','Track Damage Cost',
                           'Total Damage Cost','Total Persons Injured',
                           'Total Persons Killed','Temperature']
               if c in df.columns]

rows = []
for col in key_numeric:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_iqr = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
    n_z   = (np.abs(zscore(s)) > 3).sum()
    rows.append({'Признак': col,
                 'IQR выбросы (%)': round(100*n_iqr/len(s), 2),
                 'Z-score выбросы (%)': round(100*n_z/len(s), 2)})

outlier_df = pd.DataFrame(rows)
print(outlier_df.to_string(index=False))


              Признак  IQR выбросы (%)  Z-score выбросы (%)
          Train Speed            12.08                 2.01
        Maximum Speed            10.90                 1.61
        Gross Tonnage             3.31                 1.16
Equipment Damage Cost            12.68                 1.17
    Track Damage Cost            12.56                 0.93
    Total Damage Cost            13.28                 1.15
Total Persons Injured             5.74                 0.17
 Total Persons Killed             1.27                 1.27
          Temperature             0.22                 0.19


## 2.4. Кодирование категориальных переменных

Применяем **One-Hot Encoding** для ключевых категориальных признаков.


In [264]:
cat_to_encode = [c for c in ['Accident Type','Weather Condition','Visibility',
                              'Track Type','Train Direction','Equipment Type']
                 if c in df.columns]

for col in cat_to_encode:
    print(f"{col}: {df[col].nunique()} уникальных значений — {df[col].value_counts().index[:5].tolist()}")


Train Direction: 4 уникальных значений — ['East', 'West', 'North', 'South']
Equipment Type: 14 уникальных значений — ['Freight Train', 'Yard/switching', 'Cut of cars', 'Light loco(s)', 'Single Car']


---
# 3. Разведочный анализ данных и визуализации

## 3.1. Описательная статистика числовых признаков


In [265]:
desc = df[key_numeric].describe().round(2)
print(desc.to_string())


       Train Speed  Maximum Speed  Gross Tonnage  Equipment Damage Cost  Track Damage Cost  Total Damage Cost  Total Persons Injured  Total Persons Killed  Temperature
count    181735.00      181737.00      181737.00              181737.00          181737.00          181737.00              181737.00             181737.00    181737.00
mean         12.60          13.71        3508.47               45982.78           19889.83           75114.14                   0.16                  0.02        56.08
std          15.39          15.77        4524.27              206759.91          103645.15          308219.75                   3.18                  0.22        23.32
min           0.00           0.00           0.00                   0.00               0.00               0.00                   0.00                  0.00       -65.00
25%           4.00           4.00           0.00                2500.00               0.00            8043.00                   0.00                  0.00      

## График 1. Распределение типов аварий (Bar Chart)

**Гипотеза:** сходы с рельсов (Derailment) должны составлять подавляющее большинство инцидентов.


In [266]:
if 'Accident Type' in df.columns:
    top_types = df['Accident Type'].value_counts().head(8)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(top_types.index[::-1], top_types.values[::-1], color=PALETTE[:len(top_types)])
    for bar, val in zip(bars, top_types.values[::-1]):
        ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
    ax.set_xlabel('Количество аварий')
    ax.set_title('Рис. 1. Топ-8 типов железнодорожных аварий', fontweight='bold', pad=12)
    plt.tight_layout()
    plt.savefig('fig01_accident_types.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    print("\n Вывод: Сходы с рельсов (Derailment) составляют более 70% всех инцидентов,")
    print("что делает их главным объектом анализа. Столкновения (Other impacts) —")
    print("второй по частоте тип аварий (~8%).")



 Вывод: Сходы с рельсов (Derailment) составляют более 70% всех инцидентов,
что делает их главным объектом анализа. Столкновения (Other impacts) —
второй по частоте тип аварий (~8%).


## График 2. Распределение общего ущерба (Total Damage Cost)

**Гипотеза:** распределение ущерба сильно скошено вправо — большинство аварий дешёвые, но редкие катастрофы огромны.


In [267]:
if 'Total Damage Cost' in df.columns:
    data = df['Total Damage Cost'].clip(upper=df['Total Damage Cost'].quantile(0.99))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(data, bins=60, color=PALETTE[0], edgecolor='white', alpha=0.85)
    axes[0].set_title('Линейная шкала')
    axes[0].set_xlabel('Ущерб ($)')
    axes[0].set_ylabel('Частота')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    log_data = np.log1p(df['Total Damage Cost'])
    axes[1].hist(log_data, bins=60, color=PALETTE[1], edgecolor='white', alpha=0.85)
    axes[1].set_title('Логарифмическая шкала (log1p)')
    axes[1].set_xlabel('log(1 + Ущерб)')
    axes[1].set_ylabel('Частота')

    fig.suptitle('Рис. 2. Распределение общего ущерба от аварий', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('fig02_damage_dist.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    median_val = df['Total Damage Cost'].median()
    mean_val = df['Total Damage Cost'].mean()
    print(f"\n Вывод: Распределение резко правосторонне скошено.")
    print(f"   Медиана: ${median_val:,.0f} | Среднее: ${mean_val:,.0f}")
    print("Логарифмическое преобразование приближает распределение к нормальному,")
    print("что критически важно для линейных моделей.")



 Вывод: Распределение резко правосторонне скошено.
   Медиана: $16,600 | Среднее: $75,114
Логарифмическое преобразование приближает распределение к нормальному,
что критически важно для линейных моделей.


## График 3. Динамика числа аварий по годам (Time Series)

**Гипотеза:** наблюдается снижение числа аварий со временем благодаря улучшению технологий и регулирования.


In [268]:
year_col = 'Year' if 'Year' in df.columns else ('Accident Year' if 'Accident Year' in df.columns else None)
if year_col:
    year_values = pd.to_numeric(df[year_col], errors='coerce')
    if year_values.dropna().max() <= 99:
        year_values = pd.Series(
            np.where(year_values.isna(), np.nan,
                     np.where(year_values <= 30, year_values + 2000, year_values + 1900)),
            index=df.index,
        )
    year_values = year_values[year_values.between(1975, 2025)]
    yearly = year_values.astype(int).value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.fill_between(yearly.index, yearly.values, alpha=0.25, color=PALETTE[0])
    ax.plot(yearly.index, yearly.values, color=PALETTE[0], linewidth=2.2, marker='o', markersize=3)
    ax.set_xlabel('Год')
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 3. Количество железнодорожных аварий по годам (1975–2025)',
                 fontweight='bold', pad=12)
    # Аннотация тренда
    z = np.polyfit(yearly.index, yearly.values, 1)
    p = np.poly1d(z)
    ax.plot(yearly.index, p(yearly.index), '--', color=PALETTE[1], linewidth=1.5, label='Тренд')
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig03_yearly_accidents.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    print("\n Вывод: Наблюдается выраженный нисходящий тренд — число аварий")
    print("снизилось с пика 1970–80-х до минимумов 2010–2020-х годов.")
    print("Это отражает улучшение стандартов безопасности и технологий.")



 Вывод: Наблюдается выраженный нисходящий тренд — число аварий
снизилось с пика 1970–80-х до минимумов 2010–2020-х годов.
Это отражает улучшение стандартов безопасности и технологий.


## График 4. Сезонность аварий по месяцам


In [269]:
month_col = 'Accident Month' if 'Accident Month' in df.columns else None
if month_col and month_col in df.columns:
    monthly = df[month_col].value_counts().sort_index()
    months_ru = ['Янв','Фев','Мар','Апр','Май','Июн',
                 'Июл','Авг','Сен','Окт','Ноя','Дек']

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = [PALETTE[1] if v == monthly.max() else PALETTE[0] for v in monthly.values]
    ax.bar(range(1, 13), monthly.values, color=colors, edgecolor='white')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(months_ru)
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 4. Сезонность аварий по месяцам', fontweight='bold', pad=12)
    ax.axhline(monthly.mean(), color='gray', linestyle='--', label=f'Среднее: {monthly.mean():.0f}')
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig04_monthly.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    peak = months_ru[monthly.idxmax()-1]
    print(f"\n Вывод: Пик аварийности приходится на {peak} месяц.")
    print("Зимние месяцы характеризуются ростом числа инцидентов, что связано")
    print("с неблагоприятными погодными условиями (снег, лёд, мороз).")



 Вывод: Пик аварийности приходится на Мар месяц.
Зимние месяцы характеризуются ростом числа инцидентов, что связано
с неблагоприятными погодными условиями (снег, лёд, мороз).


## График 5. Аварии по типам пути (Track Type)


In [270]:
if 'Track Type' in df.columns:
    track_data = df['Track Type'].value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].barh(track_data.index, track_data.values,
                 color=PALETTE[:len(track_data)])
    axes[0].set_xlabel('Число аварий')
    axes[0].set_title('По типу пути')

    axes[1].pie(track_data.values, labels=track_data.index,
                autopct='%1.1f%%', colors=PALETTE[:len(track_data)],
                startangle=90)
    axes[1].set_title('Доля по типу пути')

    fig.suptitle('Рис. 5. Распределение аварий по типу пути', fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig05_track_type.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    top_track = track_data.idxmax()
    print(f"\n Вывод: Наибольшее количество аварий происходит на путях типа '{top_track}'.")
    print("Это важно для приоритизации инспекций и технического обслуживания.")



 Вывод: Наибольшее количество аварий происходит на путях типа 'Yard'.
Это важно для приоритизации инспекций и технического обслуживания.


## График 6. Ущерб по типу аварии (Boxplot)

**Гипотеза:** столкновения наносят больший ущерб, чем сходы с рельсов.


In [271]:
if 'Accident Type' in df.columns and 'Total Damage Cost' in df.columns:
    top4 = df['Accident Type'].value_counts().head(4).index
    sub = df[df['Accident Type'].isin(top4)].copy()
    sub['log_damage'] = np.log1p(sub['Total Damage Cost'])

    fig, ax = plt.subplots(figsize=(11, 5))
    sub.boxplot(column='log_damage', by='Accident Type', ax=ax,
                notch=True, patch_artist=True,
                boxprops=dict(facecolor=PALETTE[0], alpha=0.6),
                medianprops=dict(color=PALETTE[1], linewidth=2),
                flierprops=dict(marker='.', alpha=0.2))
    ax.set_xlabel('Тип аварии')
    ax.set_ylabel('log(1 + Ущерб $)')
    ax.set_title('Рис. 6. Распределение ущерба по типу аварии', fontweight='bold', pad=12)
    plt.suptitle('')
    plt.tight_layout()
    plt.savefig('fig06_damage_by_type.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    print("\n Вывод: Медианный ущерб существенно различается между типами аварий.")
    print("Боксплоты показывают наличие экстремальных выбросов во всех категориях —")
    print("редкие крупные катастрофы сильно превышают типичный уровень ущерба.")



 Вывод: Медианный ущерб существенно различается между типами аварий.
Боксплоты показывают наличие экстремальных выбросов во всех категориях —
редкие крупные катастрофы сильно превышают типичный уровень ущерба.


## График 7. Корреляционная матрица числовых признаков (Heatmap)

**Цель:** выявить мультиколлинеарность перед построением регрессионных моделей.


In [272]:
corr_cols = [c for c in ['Train Speed','Maximum Speed','Gross Tonnage',
                           'Equipment Damage Cost','Track Damage Cost',
                           'Total Damage Cost','Total Persons Injured',
                           'Total Persons Killed','Temperature',
                           'Head End Locomotives','Loaded Freight Cars',
                           'Empty Freight Cars']
             if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 8}, linewidths=0.5)
ax.set_title('Рис. 7. Матрица корреляций числовых признаков', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig07_corr_heatmap.png', bbox_inches='tight')
show_current_matplotlib_figure()

# Топ корреляции
pairs = [(c1, c2, corr_matrix.loc[c1,c2])
         for c1, c2 in combinations(corr_cols, 2)]
pairs_sorted = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)
print("\n Топ-5 наиболее скоррелированных пар признаков:")
for c1, c2, r in pairs_sorted[:5]:
    print(f"   {c1}  ↔  {c2}: r = {r:.3f}")
print("\n Высокая корреляция между Equipment Damage Cost и Total Damage Cost")
print("означает мультиколлинеарность — учтём при построении моделей.")



 Топ-5 наиболее скоррелированных пар признаков:
   Train Speed  ↔  Maximum Speed: r = 0.922
   Equipment Damage Cost  ↔  Total Damage Cost: r = 0.835
   Gross Tonnage  ↔  Loaded Freight Cars: r = 0.783
   Track Damage Cost  ↔  Total Damage Cost: r = 0.572
   Head End Locomotives  ↔  Loaded Freight Cars: r = 0.435

 Высокая корреляция между Equipment Damage Cost и Total Damage Cost
означает мультиколлинеарность — учтём при построении моделей.


## График 8. Scatter: Скорость поезда vs Ущерб

**Гипотеза:** чем выше скорость, тем больше ущерб от аварии.


In [273]:
if 'Train Speed' in df.columns and 'Total Damage Cost' in df.columns:
    sub = df[(df['Train Speed'] > 0) & (df['Total Damage Cost'] > 0)].sample(
        min(5000, len(df)), random_state=42)
    log_damage = np.log1p(sub['Total Damage Cost'])
    speed = sub['Train Speed']

    # Линия регрессии
    z = np.polyfit(speed, log_damage, 1)
    p = np.poly1d(z)
    r, pval = pearsonr(speed, log_damage)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(speed, log_damage, alpha=0.3, s=15, color=PALETTE[0])
    x_line = np.linspace(speed.min(), speed.max(), 200)
    ax.plot(x_line, p(x_line), color=PALETTE[1], linewidth=2.2,
            label=f'Регрессия (r={r:.3f}, p={pval:.3e})')
    ax.set_xlabel('Скорость поезда (миль/ч)')
    ax.set_ylabel('log(1 + Ущерб $)')
    ax.set_title('Рис. 8. Зависимость ущерба от скорости поезда', fontweight='bold', pad=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig08_speed_damage.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    print(f"\n Вывод: Коэффициент корреляции Пирсона r = {r:.3f}.")
    print("Более высокая скорость статистически связана с большим ущербом,")
    print("хотя связь умеренная — ущерб определяется многими факторами.")



 Вывод: Коэффициент корреляции Пирсона r = 0.332.
Более высокая скорость статистически связана с большим ущербом,
хотя связь умеренная — ущерб определяется многими факторами.


## График 9. Аварии по штатам (топ-15)


In [274]:
if 'State Name' in df.columns:
    top_states = df['State Name'].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = [PALETTE[0] if i > 0 else PALETTE[1] for i in range(len(top_states))]
    bars = ax.barh(top_states.index[::-1], top_states.values[::-1], color=colors[::-1])
    for bar, val in zip(bars, top_states.values[::-1]):
        ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
    ax.set_xlabel('Число аварий')
    ax.set_title('Рис. 9. Топ-15 штатов по числу аварий', fontweight='bold', pad=12)
    plt.tight_layout()
    plt.savefig('fig09_states.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    print(f"\n Вывод: Наибольшее число аварий — в штате {top_states.index[0]}.")
    print("Это может объясняться протяжённостью сети, объёмом перевозок и")
    print("климатическими условиями данного региона.")



 Вывод: Наибольшее число аварий — в штате TEXAS.
Это может объясняться протяжённостью сети, объёмом перевозок и
климатическими условиями данного региона.


## График 10. Аварии по погодным условиям


In [275]:
if 'Weather Condition' in df.columns:
    weather = df['Weather Condition'].value_counts()

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(weather.index, weather.values, color=PALETTE[:len(weather)], edgecolor='white')
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 10. Распределение аварий по погодным условиям',
                 fontweight='bold', pad=12)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('fig10_weather.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    clear_pct = 100 * weather.get('Clear', 0) / weather.sum()
    print(f"\n Вывод: {clear_pct:.1f}% аварий происходит при ясной погоде.")
    print("Это не означает, что ясная погода опасна — она просто самая частая.")
    print("Нормированный показатель (аварии / общий трафик) дал бы другую картину.")



 Вывод: 64.1% аварий происходит при ясной погоде.
Это не означает, что ясная погода опасна — она просто самая частая.
Нормированный показатель (аварии / общий трафик) дал бы другую картину.


---
# Регрессионный анализ

---

## Что такое регрессия и зачем она нужна?

**Регрессия** — это метод, который находит математическую связь между
**входными признаками** (X) и **целевой переменной** (y).

| Тип | Цель | Пример в нашем проекте |
|-----|------|------------------------|
| **Линейная регрессия** | Предсказать число | Сколько $ ущерба от аварии? |
| **Логистическая регрессия** | Класс 0 или 1 | Будут ли жертвы? |
| **Мультиномиальная** | Один из K классов | Тип пути: Main/Yard/Industry? |

---

## Линейная регрессия — основы

### Формула модели:
```
ŷ = β₀ + β₁·x₁ + β₂·x₂ + ... + βₖ·xₖ
```
- **ŷ** — прогноз (у нас: логарифм ущерба)
- **β₀** — свободный член (значение y когда все x = 0)
- **β₁...βₖ** — коэффициенты: насколько меняется y при +1 единице xᵢ
- **x₁...xₖ** — признаки (скорость, тоннаж, температура...)

### Как обучается — Метод Наименьших Квадратов (OLS):
Модель ищет коэффициенты β, минимизируя сумму квадратов ошибок:
```
min Σ(yᵢ - ŷᵢ)²
```
Аналитическое решение: **β̂ = (XᵀX)⁻¹Xᵀy**

### Метрики качества — как читать результаты:

| Метрика | Формула | Что означает | Лучше когда |
|---------|---------|--------------|-------------|
| **R²** | 1 − SS_res/SS_tot | Доля объяснённой дисперсии | Ближе к 1 |
| **RMSE** | √(Σ(y−ŷ)²/n) | Средняя квадратичная ошибка | Меньше |
| **MAE** | Σ|y−ŷ|/n | Средняя абсолютная ошибка | Меньше |

**R² = 0.3** → модель объясняет 30% разброса (остальное — шум/неизвестные факторы)
**R² = 0.8** → хорошо: 80% объяснено

> **Важно:** После `StandardScaler` все признаки имеют μ=0, σ=1.
> Тогда β = 0.5 означает: «рост признака на 1σ → рост y на 0.5σ»

---

## OLS через statsmodels — что читать в сводке

| Показатель | Что читать |
|------------|-----------|
| **coef** | Значение коэффициента β |
| **std err** | Стандартная ошибка — точность оценки β |
| **t / P>|t|** | t-статистика и p-значение: p < 0.05 → признак значим |
| **[0.025, 0.975]** | 95% доверительный интервал для β |
| **R-squared** | Доля объяснённой дисперсии |
| **F-statistic** | Тест значимости всей модели (p < 0.05 → модель работает) |
| **AIC / BIC** | Качество модели с штрафом за сложность (меньше = лучше) |

### Q-Q Plot остатков:
Линейная регрессия предполагает, что остатки ε ~ N(0, σ²).
На Q-Q Plot: точки вдоль диагонали → нормальность подтверждена.

---

## Регуляризация: Ridge и Lasso

**Проблема OLS:** при многих признаках возможно переобучение.
**Решение:** добавить штраф за большие коэффициенты.

### Ridge (L2):
```
min { Σ(y−ŷ)² + λ·Σβⱼ² }
```
→ Уменьшает ВСЕ коэффициенты, но не обнуляет. Хорош при мультиколлинеарности.

### Lasso (L1):
```
min { Σ(y−ŷ)² + λ·Σ|βⱼ| }
```
→ **Обнуляет** малозначимые β — встроенный отбор признаков!

**λ (alpha в sklearn):** чем больше → тем сильнее штраф → проще модель.

---

## Логистическая регрессия — классификация через вероятности

Несмотря на слово "регрессия" — это **алгоритм классификации**.

### Сигмоидная функция:
```
P(Y=1|x) = σ(z) = 1 / (1 + e^(-z))    где z = β₀ + β₁x₁ + ...
```
Выход — **вероятность** принадлежности к классу 1.
Если P > 0.5 → предсказываем класс 1, иначе класс 0.

### Матрица ошибок (Confusion Matrix):
```
                 Предсказано: 0    Предсказано: 1
Реальный: 0   [  TN (верно!)       FP (ложная тревога) ]
Реальный: 1   [  FN (пропуск!)     TP (верно!)         ]
```
**Диагональ (TN, TP)** — правильные ответы → хотим максимизировать
**FN** — пропущенные жертвы → самая опасная ошибка в задачах безопасности!

### Метрики классификации:

| Метрика | Формула | Смысл |
|---------|---------|-------|
| **Accuracy** | (TP+TN)/(P+N) | Доля правильных ответов |
| **Precision** | TP/(TP+FP) | Из предсказанных "1" — сколько верных |
| **Recall** | TP/(TP+FN) | Из реальных "1" — сколько нашли |
| **F1-score** | 2·P·R/(P+R) | Баланс Precision и Recall |
| **ROC-AUC** | Площадь под ROC | 0.5=случайно, 1.0=идеально |

### Несбалансированные классы — частая проблема:
Если жертвы в 5% случаев → наивная модель: "нет жертв" всегда → Accuracy = 95%
→ **бесполезная модель!**
**Решение:** `class_weight='balanced'` — увеличивает штраф за ошибки на редком классе.

---

## Мультиномиальная логистическая регрессия (K классов)

Расширение на **K > 2** классов через функцию **Softmax**:
```
P(Y=k|x) = exp(xᵀβₖ) / Σⱼ exp(xᵀβⱼ)
```
Для каждого класса — своя линейная комбинация, softmax нормирует в вероятности (сумма = 1).

---

## Как выбрать лучшую модель?

1. **R² / AUC** — основной показатель качества
2. **Cross-validation (CV)** — честная оценка, не переобучение
3. **Кривая обучения** — train >> validation → переобучение
4. **F1 vs Accuracy** — при дисбалансе классов F1 важнее Accuracy
5. **Бизнес-задача** — Recall важнее Precision при высокой цене FN

---

---
# 4. Модели регрессии и классификации

## 4.1. Формулировка задачи

Мы рассматриваем три типа регрессионных задач:

| Задача | Тип | Целевая переменная |
|--------|-----|--------------------|
| 1 | **Линейная регрессия (множественная)** | `log(Total Damage Cost)` — непрерывная |
| 2 | **Логистическая регрессия (бинарная)** | Есть/нет пострадавших (`has_injury`) |
| 3 | **Мультиномиальная логистическая регрессия** | Тип пути: Main / Yard / Industry |

---

## 4.2. Линейная (множественная) регрессия

### Теоретическая база

Модель множественной линейной регрессии:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_k x_k + \varepsilon$$

Где:
- $\hat{y}$ — прогнозируемое значение (логарифм ущерба)
- $\beta_0$ — свободный член (intercept)
- $\beta_i$ — коэффициенты при каждом признаке
- $x_i$ — значения признаков (скорость, тоннаж и т.д.)
- $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ — случайная ошибка

**Метод оценки коэффициентов** — Метод наименьших квадратов (OLS):

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Метрики качества:**
- $R^2 = 1 - \dfrac{SS_{res}}{SS_{tot}}$ — доля объяснённой дисперсии
- $RMSE = \sqrt{\dfrac{1}{n}\sum(y_i - \hat{y}_i)^2}$ — средняя квадратичная ошибка
- $MAE = \dfrac{1}{n}\sum|y_i - \hat{y}_i|$ — средняя абсолютная ошибка


In [276]:
model_df = enrich_operational_features(df)

reg_df = model_df[model_df['Total Damage Cost'].fillna(0) > 0].copy()
reg_df['log_total_damage'] = np.log1p(reg_df['Total Damage Cost'])

reg_num_features = [
    c for c in [
        'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'log_Gross Tonnage', 'Temperature',
        'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars', 'Total Freight Cars',
        'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars', 'Total Derailed Units',
        'Track Class', 'Track Density', 'Speed Utilization', 'Derailed Share'
    ] if c in reg_df.columns
]
reg_cat_features = [
    c for c in [
        'Accident Type', 'Weather Condition', 'Visibility', 'Track Type',
        'Train Direction', 'Equipment Type', 'Signalization', 'Method Of Operation'
    ] if c in reg_df.columns
]

X_reg = reg_df[reg_num_features + reg_cat_features].copy()
y_reg = reg_df['log_total_damage'].copy()

reg_preprocessor = make_preprocessor(reg_num_features, reg_cat_features, scale_numeric=True)

X_reg_train_raw, X_reg_test_raw, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

X_train = reg_preprocessor.fit_transform(X_reg_train_raw)
X_test = reg_preprocessor.transform(X_reg_test_raw)
y_train = y_reg_train
y_test = y_reg_test
reg_feature_names = reg_preprocessor.get_feature_names_out()

leakage_features = [c for c in ['Equipment Damage Cost', 'Track Damage Cost'] if c in df.columns]

print('Подготовка данных для модели ущерба')
print(f'  Строк с положительным ущербом: {len(reg_df):,}')
print(f'  Обучающая выборка: {X_train.shape[0]:,}')
print(f'  Тестовая выборка:  {X_test.shape[0]:,}')
print()
print('Что именно проверяем:')
print('  Можно ли по характеристикам поезда, пути и типа аварии заранее объяснить размер ущерба.')
print('  Целевая переменная: log(1 + Total Damage Cost).')
print()
print('Почему этот набор признаков лучше предыдущего:')
print('  Мы специально НЕ используем Equipment Damage Cost и Track Damage Cost.')
print('  Эти величины входят в Total Damage Cost и давали бы модели почти готовый ответ.')
if leakage_features:
    print(f'  Исключены признаки с утечкой цели: {leakage_features}')
print()
print(f'Числовые признаки: {reg_num_features}')
print(f'Категориальные признаки: {reg_cat_features}')


Подготовка данных для модели ущерба
  Строк с положительным ущербом: 181,585
  Обучающая выборка: 145,268
  Тестовая выборка:  36,317

Что именно проверяем:
  Можно ли по характеристикам поезда, пути и типа аварии заранее объяснить размер ущерба.
  Целевая переменная: log(1 + Total Damage Cost).

Почему этот набор признаков лучше предыдущего:
  Мы специально НЕ используем Equipment Damage Cost и Track Damage Cost.
  Эти величины входят в Total Damage Cost и давали бы модели почти готовый ответ.
  Исключены признаки с утечкой цели: ['Equipment Damage Cost', 'Track Damage Cost']

Числовые признаки: ['Train Speed', 'Maximum Speed', 'Gross Tonnage', 'log_Gross Tonnage', 'Temperature', 'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars', 'Total Freight Cars', 'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars', 'Total Derailed Units', 'Track Class', 'Track Density', 'Speed Utilization', 'Derailed Share']
Категориальные признаки: ['Accident Type', 'Weather Condition

In [277]:
reg_models = {
    'OLS (без штрафа)': LinearRegression(),
    'Ridge (стабильная)': RidgeCV(alphas=np.logspace(-3, 3, 10)),
    'ElasticNet (отбор)': ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-3, 1, 8),
        max_iter=5000,
        cv=3,
        random_state=42
    ),
}

trained_reg_models = {}
results_reg = {}

for name, model in reg_models.items():
    model.fit(X_train, y_train)
    trained_reg_models[name] = model
    y_pred = model.predict(X_test)
    cv_pipe = Pipeline([
        ('preprocessor', make_preprocessor(reg_num_features, reg_cat_features, scale_numeric=True)),
        ('model', model)
    ])
    cv_r2 = cross_val_score(cv_pipe, X_reg, y_reg, cv=3, scoring='r2', n_jobs=1)
    results_reg[name] = {
        'R² (test)': round(r2_score(y_test, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_test, y_pred), 4),
        'CV R² mean': round(cv_r2.mean(), 4),
        'CV R² std': round(cv_r2.std(), 4),
    }

results_df = pd.DataFrame(results_reg).T.sort_values('CV R² mean', ascending=False)
best_candidate = results_df.index[0]
ridge_gap = abs(results_df.loc['Ridge (стабильная)', 'CV R² mean'] - results_df.iloc[0]['CV R² mean'])
reg_model_name = 'Ridge (стабильная)' if ridge_gap <= 0.005 else best_candidate
lr = trained_reg_models[reg_model_name]
y_pred_lr = lr.predict(X_test)

r2 = r2_score(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae = mean_absolute_error(y_test, y_pred_lr)

coef_series = pd.Series(lr.coef_, index=reg_feature_names)
coef_filtered = coef_series[~coef_series.index.str.contains('infrequent_sklearn')]
coef_df = (
    coef_filtered.rename_axis('raw_feature')
    .reset_index(name='Коэффициент')
    .assign(**{
        'Признак': lambda d: d['raw_feature'].map(pretty_feature_name),
        'Абс. значение': lambda d: d['Коэффициент'].abs()
    })
    .sort_values('Абс. значение', ascending=False)
)

ridge_coef_display_df = coef_df[
    ~coef_df['Признак'].str.contains('Class =', regex=False)
].head(12).copy()
coef_compare_features = ridge_coef_display_df['Признак'].tolist()[:8]

print('=' * 70)
print('РЕГРЕССИЯ ДЛЯ ПРОГНОЗА УЩЕРБА')
print('=' * 70)
print(results_df.to_string())
print()
print(f'Выбранная модель: {reg_model_name}')
print(f'  R² на тесте: {r2:.4f}')
print(f'  RMSE:        {rmse:.4f}')
print(f'  MAE:         {mae:.4f}')
print()
print('Как это читать без математики:')
print(f'  Модель объясняет примерно {r2*100:.1f}% различий в размере ущерба.')
print('  Это не идеальный результат, но он реалистичен для аварийных данных:')
print('  на итоговый ущерб влияют и неполно наблюдаемые факторы, и редкие экстремальные события.')
print(f'  Типичная ошибка по RMSE в лог-шкале соответствует примерно множителю x{np.exp(rmse):.2f} в реальных долларах.')
print()
print('Главный вывод по моделям:')
print('  OLS, Ridge и ElasticNet дали почти одинаковое качество.')
print('  Поэтому разумно выбрать Ridge: она остаётся интерпретируемой и устойчивее к шуму и мультиколлинеарности.')
print()
print('Самые заметные факторы у выбранной модели:')
print(ridge_coef_display_df[['Признак', 'Коэффициент']].to_string(index=False))


РЕГРЕССИЯ ДЛЯ ПРОГНОЗА УЩЕРБА
                    R² (test)    RMSE     MAE  CV R² mean  CV R² std
ElasticNet (отбор)     0.2747  1.1491  0.8997      0.2515     0.0180
OLS (без штрафа)       0.2746  1.1491  0.8997      0.2513     0.0180
Ridge (стабильная)     0.2746  1.1491  0.8997      0.2511     0.0182

Выбранная модель: Ridge (стабильная)
  R² на тесте: 0.2746
  RMSE:        1.1491
  MAE:         0.8997

Как это читать без математики:
  Модель объясняет примерно 27.5% различий в размере ущерба.
  Это не идеальный результат, но он реалистичен для аварийных данных:
  на итоговый ущерб влияют и неполно наблюдаемые факторы, и редкие экстремальные события.
  Типичная ошибка по RMSE в лог-шкале соответствует примерно множителю x3.16 в реальных долларах.

Главный вывод по моделям:
  OLS, Ridge и ElasticNet дали почти одинаковое качество.
  Поэтому разумно выбрать Ridge: она остаётся интерпретируемой и устойчивее к шуму и мультиколлинеарности.

Самые заметные факторы у выбранной модели:
   

## График 11. Коэффициенты линейной регрессии


In [278]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = ridge_coef_display_df.iloc[::-1]
colors = [PALETTE[1] if c < 0 else PALETTE[0] for c in plot_df['Коэффициент']]
ax.barh(plot_df['Признак'], plot_df['Коэффициент'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Стандартизованный коэффициент')
ax.set_title('Рис. 11. Какие факторы сильнее всего связаны с ущербом', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig11_lr_coefs.png', bbox_inches='tight')
show_current_matplotlib_figure()

print()
print('Подробный вывод:')
print('  Этот график показывает не просто корреляцию, а вклад каждого признака при фиксированных остальных.')
print('  Самые сильные положительные коэффициенты означают: чем больше таких признаков, тем выше ожидаемый ущерб.')
print('  В нашей модели особенно выделяются число сошедших с рельсов гружёных вагонов, максимально допустимая скорость')
print('  на участке и тоннаж поезда. Для обычного читателя это можно читать так: тяжёлый поезд и более серьёзный сход')
print('  почти всегда делают аварию дороже.')



Подробный вывод:
  Этот график показывает не просто корреляцию, а вклад каждого признака при фиксированных остальных.
  Самые сильные положительные коэффициенты означают: чем больше таких признаков, тем выше ожидаемый ущерб.
  В нашей модели особенно выделяются число сошедших с рельсов гружёных вагонов, максимально допустимая скорость
  на участке и тоннаж поезда. Для обычного читателя это можно читать так: тяжёлый поезд и более серьёзный сход
  почти всегда делают аварию дороже.


## График 12. Реальные vs Предсказанные значения

**Диагностика:** идеальная модель даёт точки на диагонали y = x.


In [279]:
sample_idx = np.random.choice(len(y_test), min(2500, len(y_test)), replace=False)
y_t_sample = np.array(y_test)[sample_idx]
y_p_sample = y_pred_lr[sample_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_t_sample, y_p_sample, alpha=0.3, s=15, color=PALETTE[0])
mn = min(y_t_sample.min(), y_p_sample.min())
mx = max(y_t_sample.max(), y_p_sample.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='идеальное совпадение')
axes[0].set_xlabel('Реальный log(1 + ущерб)')
axes[0].set_ylabel('Прогноз модели')
axes[0].set_title(f'Реальные vs прогнозные значения, R² = {r2:.3f}')
axes[0].legend()

residuals = y_t_sample - y_p_sample
axes[1].scatter(y_p_sample, residuals, alpha=0.3, s=15, color=PALETTE[2])
axes[1].axhline(0, color='red', linewidth=2, linestyle='--')
axes[1].set_xlabel('Прогноз модели')
axes[1].set_ylabel('Остаток (реальность - прогноз)')
axes[1].set_title('График остатков')

fig.suptitle('Рис. 12. Проверка качества регрессионной модели', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig12_lr_diagnostics.png', bbox_inches='tight')
show_current_matplotlib_figure()

print()
print('Подробный вывод:')
print('  Слева мы проверяем, насколько предсказания лежат рядом с идеальной диагональю.')
print('  Чем плотнее точки к линии, тем лучше модель угадывает порядок величины ущерба.')
print('  Справа мы смотрим, есть ли систематическая ошибка. Если облако остатков хаотично распределено вокруг нуля,')
print('  значит модель ошибается по-разному, а не в одну сторону. В этом notebook картина умеренно хорошая:')
print('  модель полезна для оценки общего масштаба аварии, но не заменяет детального расследования каждого случая.')



Подробный вывод:
  Слева мы проверяем, насколько предсказания лежат рядом с идеальной диагональю.
  Чем плотнее точки к линии, тем лучше модель угадывает порядок величины ущерба.
  Справа мы смотрим, есть ли систематическая ошибка. Если облако остатков хаотично распределено вокруг нуля,
  значит модель ошибается по-разному, а не в одну сторону. В этом notebook картина умеренно хорошая:
  модель полезна для оценки общего масштаба аварии, но не заменяет детального расследования каждого случая.


## График 3D. Плоскость множественной линейной регрессии

**3D визуализация:** два наиболее значимых признака против целевой переменной и аппроксимирующая плоскость регрессии. Расстояние точки от плоскости = остаток.

In [280]:
plot_3d_features = [c for c in ['Maximum Speed', 'Derailed Loaded Freight Cars'] if c in reg_df.columns]
plot_3d_df = reg_df[plot_3d_features + ['log_total_damage']].dropna().copy()

reg_plane = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
reg_plane.fit(plot_3d_df[plot_3d_features], plot_3d_df['log_total_damage'])

sample_3d = plot_3d_df.sample(min(2500, len(plot_3d_df)), random_state=42)
x_name, y_name = plot_3d_features

x_grid = np.linspace(sample_3d[x_name].quantile(0.01), sample_3d[x_name].quantile(0.99), 35)
y_grid = np.linspace(sample_3d[y_name].quantile(0.01), sample_3d[y_name].quantile(0.99), 35)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
grid_df = pd.DataFrame({x_name: X_mesh.ravel(), y_name: Y_mesh.ravel()})
Z_mesh = reg_plane.predict(grid_df).reshape(X_mesh.shape)

plane_r2 = r2_score(plot_3d_df['log_total_damage'], reg_plane.predict(plot_3d_df[plot_3d_features]))

if PLOTLY_AVAILABLE:
    fig_3d = go.Figure()
    fig_3d.add_trace(go.Scatter3d(
        x=sample_3d[x_name],
        y=sample_3d[y_name],
        z=sample_3d['log_total_damage'],
        mode='markers',
        marker=dict(
            size=3,
            color=sample_3d['log_total_damage'],
            colorscale='Viridis',
            opacity=0.55,
            colorbar=dict(title='log(1 + ущерб)')
        ),
        name='Наблюдения'
    ))
    fig_3d.add_trace(go.Surface(
        x=X_mesh,
        y=Y_mesh,
        z=Z_mesh,
        colorscale='Reds',
        opacity=0.55,
        showscale=False,
        name='Плоскость регрессии'
    ))
    fig_3d.update_layout(
        title='Рис. 3D. Интерактивная плоскость множественной регрессии',
        scene=dict(
            xaxis_title=x_name,
            yaxis_title=y_name,
            zaxis_title='log(1 + ущерб)'
        ),
        width=950,
        height=700,
        margin=dict(l=0, r=0, t=60, b=0)
    )
    fig_3d.show()
else:
    from IPython import get_ipython
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    ip = get_ipython()
    if ip is not None:
        try:
            ip.run_line_magic('matplotlib', 'notebook')
        except Exception:
            pass

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(sample_3d[x_name], sample_3d[y_name], sample_3d['log_total_damage'],
               c=sample_3d['log_total_damage'], cmap='viridis', s=10, alpha=0.5)
    ax.plot_surface(X_mesh, Y_mesh, Z_mesh, alpha=0.35, color='tomato')
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_zlabel('log(1 + ущерб)')
    ax.set_title('Рис. 3D. Вращаемая плоскость регрессии')
    show_current_matplotlib_figure()

print()
print('Подробный вывод:')
print('  Это интерактивный график: в Plotly он вращается мышкой прямо в output ячейки,')
print('  а при отсутствии Plotly notebook переходит на вращаемый matplotlib-вариант.')
print(f"  Для 3D-среза выбраны признаки '{x_name}' и '{y_name}', потому что они одновременно понятны и реально влияют на ущерб.")
print('  Красная поверхность показывает упрощённую двумерную версию регрессии. Точки над плоскостью и под ней')
print('  показывают, где фактический ущерб оказался выше или ниже ожидаемого.')
print(f'  У этой двумерной плоскости R² = {plane_r2:.3f}, поэтому её нужно читать как наглядный срез модели,')
print('  а не как полный прогноз по всем признакам сразу.')



Подробный вывод:
  Это интерактивный график: в Plotly он вращается мышкой прямо в output ячейки,
  а при отсутствии Plotly notebook переходит на вращаемый matplotlib-вариант.
  Для 3D-среза выбраны признаки 'Maximum Speed' и 'Derailed Loaded Freight Cars', потому что они одновременно понятны и реально влияют на ущерб.
  Красная поверхность показывает упрощённую двумерную версию регрессии. Точки над плоскостью и под ней
  показывают, где фактический ущерб оказался выше или ниже ожидаемого.
  У этой двумерной плоскости R² = 0.209, поэтому её нужно читать как наглядный срез модели,
  а не как полный прогноз по всем признакам сразу.


## 4.3. Расширенный анализ OLS (statsmodels)

`statsmodels` даёт полную сводку регрессии: p-значения, доверительные интервалы, тесты.


In [281]:
ols_features = [
    c for c in [
        'Maximum Speed', 'Gross Tonnage', 'Temperature', 'Head End Locomotives',
        'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars',
        'Total Derailed Units', 'Speed Utilization'
    ] if c in reg_df.columns
]
ols_df = reg_df[ols_features + ['log_total_damage']].dropna().copy()
ols_df = ols_df.sample(min(8000, len(ols_df)), random_state=42)

ols_scaler = StandardScaler()
X_ols_scaled = ols_scaler.fit_transform(ols_df[ols_features])
X_ols = sm.add_constant(X_ols_scaled)
y_ols = ols_df['log_total_damage'].values

ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())

print()
print('Что проверяем в OLS:')
print('  Здесь мы берём только числовые признаки и смотрим классическую статистическую сводку.')
print('  Она нужна не ради более высокой точности, а чтобы понять знак коэффициентов и их значимость.')
print('  Для обычного читателя смысл такой: если p-значение маленькое, признак стабильно связан с ущербом,')
print('  а если большое, его вклад на фоне остальных уже не так уверен.')


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.273
Model:                            OLS   Adj. R-squared:                  0.273
Method:                 Least Squares   F-statistic:                     375.8
Date:                Thu, 09 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:27:32   Log-Likelihood:                -12492.
No. Observations:                8000   AIC:                         2.500e+04
Df Residuals:                    7991   BIC:                         2.506e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          9.9301      0.013    769.741      0.0

## График 13. Q-Q Plot остатков OLS

**Цель:** проверить нормальность распределения остатков — ключевое условие OLS.


In [282]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

residuals_ols = ols_model.resid
# Q-Q Plot
sm.qqplot(residuals_ols, line='s', ax=axes[0], alpha=0.5,
          markerfacecolor=PALETTE[0], markeredgecolor='none')
axes[0].set_title('Q-Q Plot остатков')

# Histogram of residuals
axes[1].hist(residuals_ols, bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Остаток')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение остатков')

fig.suptitle('Рис. 13. Анализ нормальности остатков OLS', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig13_ols_residuals.png', bbox_inches='tight')
show_current_matplotlib_figure()

stat, p = shapiro(residuals_ols[:500])
print(f"\n Тест Шапиро-Уилка: W = {stat:.4f}, p = {p:.4e}")
print("H₀: остатки нормально распределены")
if p < 0.05:
    print("H₀ отвергается при α=0.05 — отклонение от нормальности")
    print("(характерно для больших выборок — CLT всё равно обеспечивает асимптотику)")
else:
    print("H₀ не отвергается — нормальность остатков подтверждена")



 Тест Шапиро-Уилка: W = 0.9789, p = 1.2237e-06
H₀: остатки нормально распределены
H₀ отвергается при α=0.05 — отклонение от нормальности
(характерно для больших выборок — CLT всё равно обеспечивает асимптотику)


## 4.4. Регуляризованные регрессии: Ridge и Lasso

**Ridge (L2):** добавляет штраф $\lambda \sum \beta_j^2$ — уменьшает все коэффициенты

**Lasso (L1):** добавляет штраф $\lambda \sum |\beta_j|$ — обнуляет малозначимые коэффициенты (отбор признаков)

Общая форма регуляризованной задачи:
$$\min_{\beta} \left\{ \sum_{i=1}^n (y_i - \hat{y}_i)^2 + \lambda \cdot \text{penalty}(\boldsymbol{\beta}) \right\}$$


In [283]:
print('Сравнение моделей регрессии:')
print(results_df.to_string())
print()
print('Как читать таблицу:')
print('  R² (test)  — качество на отложенной тестовой выборке.')
print('  CV R² mean — среднее качество по кросс-валидации.')
print('  CV R² std  — насколько результат плавает между разными подвыборками.')
print()
print('Вывод по выбору модели:')
best_cv_name = results_df.index[0]
print(f'  Формально лучший по среднему CV: {best_cv_name}.')
print(f'  Но итоговый рабочий выбор в notebook — {reg_model_name}, потому что она')
print('  сохраняет интерпретируемость и не проигрывает по качеству настолько, чтобы это было важно на практике.')


Сравнение моделей регрессии:
                    R² (test)    RMSE     MAE  CV R² mean  CV R² std
ElasticNet (отбор)     0.2747  1.1491  0.8997      0.2515     0.0180
OLS (без штрафа)       0.2746  1.1491  0.8997      0.2513     0.0180
Ridge (стабильная)     0.2746  1.1491  0.8997      0.2511     0.0182

Как читать таблицу:
  R² (test)  — качество на отложенной тестовой выборке.
  CV R² mean — среднее качество по кросс-валидации.
  CV R² std  — насколько результат плавает между разными подвыборками.

Вывод по выбору модели:
  Формально лучший по среднему CV: ElasticNet (отбор).
  Но итоговый рабочий выбор в notebook — Ridge (стабильная), потому что она
  сохраняет интерпретируемость и не проигрывает по качеству настолько, чтобы это было важно на практике.


In [284]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

feature_lookup = {pretty_feature_name(name): name for name in reg_feature_names}
selected_raw_features = [feature_lookup[name] for name in coef_compare_features if name in feature_lookup]

for ax, (name, model) in zip(axes, trained_reg_models.items()):
    coef_series_model = pd.Series(model.coef_, index=reg_feature_names)
    plot_values = coef_series_model[selected_raw_features]
    colors = [PALETTE[1] if c < 0 else PALETTE[0] for c in plot_values]
    ax.barh([pretty_feature_name(x) for x in selected_raw_features], plot_values, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Коэффициент')

fig.suptitle('Рис. 14. Как регуляризация меняет коэффициенты модели', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig14_ridge_lasso.png', bbox_inches='tight')
show_current_matplotlib_figure()

print()
print('Подробный вывод:')
print('  OLS оставляет коэффициенты почти без ограничений.')
print('  Ridge аккуратно уменьшает их по модулю и тем самым делает модель спокойнее и устойчивее.')
print('  ElasticNet сильнее давит слабые эффекты и частично выполняет отбор признаков.')
print('  Если говорить совсем просто: регуляризация помогает модели меньше подстраиваться под шум и лучше переноситься на новые данные.')



Подробный вывод:
  OLS оставляет коэффициенты почти без ограничений.
  Ridge аккуратно уменьшает их по модулю и тем самым делает модель спокойнее и устойчивее.
  ElasticNet сильнее давит слабые эффекты и частично выполняет отбор признаков.
  Если говорить совсем просто: регуляризация помогает модели меньше подстраиваться под шум и лучше переноситься на новые данные.


## 4.5. Логистическая регрессия: прогноз аварии с ущербом выше типичного уровня

Здесь мы используем более понятную и честную цель:
`high_damage = 1`, если ущерб **выше медианы по датасету**, и `0` в противном случае.

**Что проверяем:** можно ли по характеристикам поезда, инфраструктуры и типу аварии понять,
будет ли ущерб выше обычного уровня.

**Почему эта постановка лучше для notebook:**
- классы почти сбалансированы, поэтому матрица ошибок читается гораздо честнее;
- цель понятна обычному читателю;
- модель не пытается угадывать очень редкое событие, а решает реально интерпретируемую задачу.


In [285]:
cls_df = enrich_operational_features(df)
cls_df = cls_df[cls_df['Total Damage Cost'].fillna(0) > 0].copy()

binary_threshold = cls_df['Total Damage Cost'].median()
cls_df['high_damage'] = (cls_df['Total Damage Cost'] >= binary_threshold).astype(int)

cls_num_features = [
    c for c in [
        'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'log_Gross Tonnage', 'Temperature',
        'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars', 'Total Freight Cars',
        'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars', 'Total Derailed Units',
        'Track Class', 'Track Density', 'Speed Utilization', 'Derailed Share'
    ] if c in cls_df.columns
]
cls_cat_features = [
    c for c in [
        'Accident Type', 'Weather Condition', 'Visibility', 'Track Type',
        'Train Direction', 'Equipment Type', 'Signalization',
        'Method Of Operation', 'Class'
    ] if c in cls_df.columns
]

X_cls = cls_df[cls_num_features + cls_cat_features].copy()
y_cls = cls_df['high_damage'].copy()

cls_preprocessor = make_preprocessor(cls_num_features, cls_cat_features, scale_numeric=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

X_tr_cls = cls_preprocessor.fit_transform(X_tr)
X_te_cls = cls_preprocessor.transform(X_te)
cls_feature_names = cls_preprocessor.get_feature_names_out()

log_reg = LogisticRegression(
    max_iter=2500,
    C=1.0,
    solver='lbfgs',
    random_state=42
)
log_reg.fit(X_tr_cls, y_tr)

y_pred_log = log_reg.predict(X_te_cls)
y_prob_log = log_reg.predict_proba(X_te_cls)[:, 1]

acc_log = accuracy_score(y_te, y_pred_log)
auc_log = roc_auc_score(y_te, y_prob_log)
f1_log = f1_score(y_te, y_pred_log)
bal_acc_log = balanced_accuracy_score(y_te, y_pred_log)
cm_b = confusion_matrix(y_te, y_pred_log)
tn, fp, fn, tp = cm_b.ravel()

log_coef_series = pd.Series(log_reg.coef_[0], index=cls_feature_names)
log_coef_filtered = log_coef_series[~log_coef_series.index.str.contains('infrequent_sklearn')]
log_coef_df = (
    log_coef_filtered.rename_axis('raw_feature')
    .reset_index(name='Коэффициент')
    .assign(**{
        'Признак': lambda d: d['raw_feature'].map(pretty_feature_name),
        'Абс. значение': lambda d: d['Коэффициент'].abs()
    })
    .sort_values('Абс. значение', ascending=False)
)

log_coef_display_df = log_coef_df[~log_coef_df['Признак'].str.contains('Class =', regex=False)].head(12).copy()

print('=' * 70)
print('БИНАРНАЯ ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ')
print('=' * 70)
print(f"Порог 'ущерб выше типичного уровня': ${binary_threshold:,.0f}")
print(f'Доля класса 1 (выше медианы): {y_cls.mean():.1%}')
print(f'Доля класса 0 (ниже медианы): {(1 - y_cls.mean()):.1%}')
print()
print('Что проверяем:')
print('  Может ли модель отделить аварии с ущербом выше типичного уровня от остальных.')
print(f'  Accuracy:            {acc_log:.4f}')
print(f'  F1-score:            {f1_log:.4f}')
print(f'  Balanced Accuracy:   {bal_acc_log:.4f}')
print(f'  ROC-AUC:             {auc_log:.4f}')
print()
print('Почему эта постановка выглядит лучше предыдущей:')
print('  Здесь классы почти 50/50, поэтому матрица ошибок не залипает в одну строку.')
print('  Мы сознательно ушли от редкого события "есть пострадавшие", чтобы получить понятную визуально и честную модель.')
print()
print('Наиболее заметные факторы у логистической модели:')
print(log_coef_display_df[['Признак', 'Коэффициент']].to_string(index=False))


БИНАРНАЯ ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
Порог 'ущерб выше типичного уровня': $16,623
Доля класса 1 (выше медианы): 50.0%
Доля класса 0 (ниже медианы): 50.0%

Что проверяем:
  Может ли модель отделить аварии с ущербом выше типичного уровня от остальных.
  Accuracy:            0.6997
  F1-score:            0.6777
  Balanced Accuracy:   0.6997
  ROC-AUC:             0.7578

Почему эта постановка выглядит лучше предыдущей:
  Здесь классы почти 50/50, поэтому матрица ошибок не залипает в одну строку.
  Мы сознательно ушли от редкого события "есть пострадавшие", чтобы получить понятную визуально и честную модель.

Наиболее заметные факторы у логистической модели:
                             Признак  Коэффициент
   Accident Type = Hwy-rail crossing    -0.584804
        Derailed Loaded Freight Cars     0.556055
       Accident Type = Other impacts     0.367112
                Total Derailed Units     0.348579
                       Gross Tonnage     0.322039
      Equipment Type = Light loco(s)     

## График 15. Матрица ошибок (Confusion Matrix)


In [286]:
cm = confusion_matrix(y_te, y_pred_log)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
fpr, tpr, _ = roc_curve(y_te, y_prob_log)
auc_val = roc_auc_score(y_te, y_prob_log)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

labels_bin = ['Ниже медианы', 'Выше медианы']

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=labels_bin, yticklabels=labels_bin,
    linewidths=0.5, linecolor='gray',
    annot_kws={'size': 14, 'weight': 'bold'}
)
axes[0].set_xlabel('Предсказанный класс', fontsize=11)
axes[0].set_ylabel('Реальный класс', fontsize=11)
axes[0].set_title('Матрица ошибок (абсолютные числа)', fontweight='bold')

annot_norm = np.array([[f'{v:.1%}' for v in row] for row in cm_norm])
sns.heatmap(
    cm_norm, annot=annot_norm, fmt='', cmap='Blues', ax=axes[1],
    xticklabels=labels_bin, yticklabels=labels_bin,
    linewidths=0.5, linecolor='gray', vmin=0, vmax=1,
    annot_kws={'size': 12}
)
axes[1].set_xlabel('Предсказанный класс', fontsize=11)
axes[1].set_ylabel('Реальный класс', fontsize=11)
axes[1].set_title('Нормированная матрица (строка = recall)', fontweight='bold')

axes[2].plot(fpr, tpr, color=PALETTE[0], linewidth=2.5, label=f'ROC (AUC = {auc_val:.3f})')
axes[2].fill_between(fpr, tpr, alpha=0.15, color=PALETTE[0])
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Случайное угадывание')
axes[2].scatter([0], [1], color='gold', s=120, zorder=5, label='Идеальная точка')
axes[2].set_xlabel('False Positive Rate', fontsize=10)
axes[2].set_ylabel('True Positive Rate', fontsize=10)
axes[2].set_title('ROC-кривая', fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].set_aspect('equal')

fig.suptitle('Рис. 15. Диагностика логистической регрессии', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig15_logistic.png', bbox_inches='tight')
show_current_matplotlib_figure()

tn_v, fp_v, fn_v, tp_v = cm.ravel()
print()
print('Подробный вывод:')
print(f'  TN = {tn_v:,}: модель верно распознала аварию как не превышающую типичный ущерб.')
print(f'  TP = {tp_v:,}: модель верно нашла аварию с ущербом выше медианы.')
print(f'  FP = {fp_v:,}: ложная тревога — модель переоценила серьёзность.')
print(f'  FN = {fn_v:,}: опасный промах — модель недооценила ущерб.')
print()
print('  Почему сейчас матрица ошибок выглядит лучше:')
print(f'    Ниже медианы: recall = {cm_norm[0, 0]:.1%}')
print(f'    Выше медианы: recall = {cm_norm[1, 1]:.1%}')
print('  То есть диагональ сильнее, чем в старой постановке с редким классом пострадавших.')
print('  Для обычного читателя это означает: модель уже не просто повторяет самый частый класс,')
print('  а действительно различает более дорогие и менее дорогие аварии.')



Подробный вывод:
  TN = 13,947: модель верно распознала аварию как не превышающую типичный ущерб.
  TP = 11,465: модель верно нашла аварию с ущербом выше медианы.
  FP = 4,211: ложная тревога — модель переоценила серьёзность.
  FN = 6,694: опасный промах — модель недооценила ущерб.

  Почему сейчас матрица ошибок выглядит лучше:
    Ниже медианы: recall = 76.8%
    Выше медианы: recall = 63.1%
  То есть диагональ сильнее, чем в старой постановке с редким классом пострадавших.
  Для обычного читателя это означает: модель уже не просто повторяет самый частый класс,
  а действительно различает более дорогие и менее дорогие аварии.


## График 16. Сигмоидная функция и интерпретация логистической регрессии


In [287]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

z = np.linspace(-6, 6, 300)
sigma = 1 / (1 + np.exp(-z))
axes[0].plot(z, sigma, color=PALETTE[0], linewidth=2.5)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.7)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.7)
axes[0].fill_between(z, 0, sigma, where=(z > 0), alpha=0.15, color=PALETTE[1], label='Вероятность > 0.5')
axes[0].fill_between(z, 0, sigma, where=(z < 0), alpha=0.15, color=PALETTE[0], label='Вероятность < 0.5')
axes[0].set_xlabel('Линейная комбинация признаков z')
axes[0].set_ylabel('P(Y = 1)')
axes[0].set_title('Как логистическая регрессия переводит признаки в вероятность')
axes[0].legend()

coef_plot_df = log_coef_display_df.iloc[::-1]
colors_log = [PALETTE[1] if c < 0 else PALETTE[0] for c in coef_plot_df['Коэффициент']]
axes[1].barh(coef_plot_df['Признак'], coef_plot_df['Коэффициент'], color=colors_log)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Коэффициент (log-odds)')
axes[1].set_title('Какие признаки сильнее двигают вероятность вверх или вниз')

fig.suptitle('Рис. 16. Интерпретация логистической регрессии', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig16_sigmoid.png', bbox_inches='tight')
show_current_matplotlib_figure()

print()
print('Подробный вывод:')
print('  Левая часть графика показывает главное правило логистической регрессии:')
print('  как только суммарный сигнал модели проходит через 0, вероятность начинает быстро смещаться к 1.')
print('  Правая часть показывает, какие признаки толкают прогноз вверх, а какие вниз.')
print('  Например, рост числа сошедших с рельсов гружёных вагонов, тоннажа и максимально допустимой скорости')
print('  делает сценарий дорогой аварии более вероятным. Отрицательные коэффициенты, наоборот, тянут риск вниз.')



Подробный вывод:
  Левая часть графика показывает главное правило логистической регрессии:
  как только суммарный сигнал модели проходит через 0, вероятность начинает быстро смещаться к 1.
  Правая часть показывает, какие признаки толкают прогноз вверх, а какие вниз.
  Например, рост числа сошедших с рельсов гружёных вагонов, тоннажа и максимально допустимой скорости
  делает сценарий дорогой аварии более вероятным. Отрицательные коэффициенты, наоборот, тянут риск вниз.


## График 17. Мультиномиальная логистическая регрессия для трёх уровней ущерба

В этой части мы делим аварии на три равные группы:
- **низкий ущерб**
- **средний ущерб**
- **высокий ущерб**

**Что проверяем:** умеет ли модель отличать по характеристикам аварии не только два класса, а сразу три уровня серьёзности.

Эта постановка полезна тем, что:
- она понятнее обычному читателю, чем прогноз точной суммы в долларах;
- классы здесь сбалансированы по определению;
- по матрице ошибок легко увидеть, где модель путает соседние уровни серьёзности.


In [288]:
multi_df = enrich_operational_features(df)
multi_df = multi_df[multi_df['Total Damage Cost'].fillna(0) > 0].copy()
multi_df['damage_band'] = pd.qcut(
    multi_df['Total Damage Cost'],
    q=3,
    labels=['Низкий', 'Средний', 'Высокий'],
    duplicates='drop'
)

if multi_df['damage_band'].nunique() != 3:
    multi_df['damage_band'] = pd.cut(
        multi_df['Total Damage Cost'],
        bins=np.quantile(multi_df['Total Damage Cost'], [0, 1/3, 2/3, 1]),
        labels=['Низкий', 'Средний', 'Высокий'],
        include_lowest=True,
        duplicates='drop'
    )

multi_num_features = cls_num_features.copy()
multi_cat_features = cls_cat_features.copy()

X_multi = multi_df[multi_num_features + multi_cat_features].copy()
y_multi = multi_df['damage_band'].copy()

multi_preprocessor = make_preprocessor(multi_num_features, multi_cat_features, scale_numeric=True)

X_tr3, X_te3, y_tr3, y_te3 = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

X_tr3_ready = multi_preprocessor.fit_transform(X_tr3)
X_te3_ready = multi_preprocessor.transform(X_te3)

multi_log_reg = LogisticRegression(max_iter=4000, solver='lbfgs', random_state=42)
multi_log_reg.fit(X_tr3_ready, y_tr3)
y_pred_mn = multi_log_reg.predict(X_te3_ready)

damage_labels = ['Низкий', 'Средний', 'Высокий']
cm_mn = confusion_matrix(y_te3, y_pred_mn, labels=damage_labels)
cm_mn_norm = cm_mn.astype('float') / cm_mn.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(
    cm_mn, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0],
    xticklabels=damage_labels, yticklabels=damage_labels,
    linewidths=0.5, linecolor='gray',
    annot_kws={'size': 12, 'weight': 'bold'}
)
axes[0].set_title('Абсолютные значения', fontweight='bold')
axes[0].set_xlabel('Предсказанный класс')
axes[0].set_ylabel('Реальный класс')

annot_mn = np.array([[f'{v:.1%}' for v in row] for row in cm_mn_norm])
sns.heatmap(
    cm_mn_norm, annot=annot_mn, fmt='', cmap='YlOrRd', ax=axes[1],
    xticklabels=damage_labels, yticklabels=damage_labels,
    vmin=0, vmax=1, linewidths=0.5, linecolor='gray',
    annot_kws={'size': 11}
)
axes[1].set_title('Нормированная матрица (строка = recall)', fontweight='bold')
axes[1].set_xlabel('Предсказанный класс')

fig.suptitle('Рис. 17. Матрица ошибок для трёх уровней ущерба', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig17_multiclass_cm.png', bbox_inches='tight')
show_current_matplotlib_figure()

acc_mn = accuracy_score(y_te3, y_pred_mn)
f1_mn = f1_score(y_te3, y_pred_mn, average='macro')

print()
print('Подробный вывод:')
print(f'  Accuracy: {acc_mn:.4f}')
print(f'  Macro F1: {f1_mn:.4f}')
print('  Матрица показывает важную и очень понятную вещь:')
print('  крайние классы Низкий и Высокий модель различает лучше, чем Средний.')
print('  Это логично: средний класс по смыслу находится между двумя соседями и естественно с ними смешивается.')
print()
for idx, label in enumerate(damage_labels):
    print(f"  Recall для класса '{label}': {cm_mn_norm[idx, idx]:.1%}")
print()
print('  Для обычного читателя вывод такой: модель уже умеет отделять дешёвые аварии от очень дорогих,')
print('  но граница вокруг среднего ущерба остаётся размытой. Это нормальный результат для реальных данных.')



Подробный вывод:
  Accuracy: 0.5436
  Macro F1: 0.5255
  Матрица показывает важную и очень понятную вещь:
  крайние классы Низкий и Высокий модель различает лучше, чем Средний.
  Это логично: средний класс по смыслу находится между двумя соседями и естественно с ними смешивается.

  Recall для класса 'Низкий': 76.8%
  Recall для класса 'Средний': 28.3%
  Recall для класса 'Высокий': 58.0%

  Для обычного читателя вывод такой: модель уже умеет отделять дешёвые аварии от очень дорогих,
  но граница вокруг среднего ущерба остаётся размытой. Это нормальный результат для реальных данных.


## График 18. Сравнение всех моделей регрессии


In [289]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

metrics_names = ['R² (test)', 'RMSE', 'MAE']
model_names = list(results_reg.keys())

for i, metric in enumerate(metrics_names):
    values = [results_reg[m][metric] for m in model_names]
    bars = axes[i].bar(model_names, values, color=PALETTE[:len(model_names)], edgecolor='white')
    for bar, val in zip(bars, values):
        axes[i].text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.01 if metric != 'R² (test)' else 0.002),
            f'{val:.4f}', ha='center', fontsize=8, fontweight='bold'
        )
    axes[i].set_title(metric, fontweight='bold')
    axes[i].set_xticks(range(len(model_names)))
    axes[i].set_xticklabels(model_names, rotation=15, ha='right', fontsize=8)

fig.suptitle('Рис. 18. Сравнение регрессионных моделей', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig18_model_comparison.png', bbox_inches='tight')
show_current_matplotlib_figure()

print()
print('Подробный вывод:')
print('  На этом графике видно, что все три линейные модели находятся очень близко друг к другу.')
print('  Это хороший признак: значит результат не держится на одной случайной настройке.')
print('  Ridge и ElasticNet почти не проигрывают обычной линейной регрессии, но дают более устойчивые коэффициенты.')
print('  Поэтому в качестве основной модели для интерпретации мы оставляем Ridge.')



Подробный вывод:
  На этом графике видно, что все три линейные модели находятся очень близко друг к другу.
  Это хороший признак: значит результат не держится на одной случайной настройке.
  Ridge и ElasticNet почти не проигрывают обычной линейной регрессии, но дают более устойчивые коэффициенты.
  Поэтому в качестве основной модели для интерпретации мы оставляем Ridge.



# 5. Анализ главных компонент (PCA)

## 5.1. Теоретическая база

**PCA** находит новые оси (главные компоненты), вдоль которых дисперсия данных максимальна:

$$\mathbf{Z} = \mathbf{X} \cdot \mathbf{W}$$

Где $\mathbf{W}$ — матрица собственных векторов ковариационной матрицы $\mathbf{C} = \frac{1}{n-1}\mathbf{X}^\top \mathbf{X}$.

**Собственное разложение:** $\mathbf{C} \mathbf{w}_j = \lambda_j \mathbf{w}_j$

Доля объяснённой дисперсии $k$-й компоненты:
$$\text{EVR}_k = \frac{\lambda_k}{\sum_{j=1}^p \lambda_j}$$


In [290]:
pca_features = [c for c in ['Train Speed','Maximum Speed','Gross Tonnage',
                            'Equipment Damage Cost','Track Damage Cost',
                            'Total Damage Cost','Temperature',
                            'Head End Locomotives','Loaded Freight Cars',
                            'Empty Freight Cars']
                if c in df.columns]

df_pca = df[pca_features].dropna()

scaler_pca = StandardScaler()
X_pca = scaler_pca.fit_transform(df_pca)

pca = PCA(random_state=42)
pca.fit(X_pca)

evr = pca.explained_variance_ratio_
cumulative_evr = np.cumsum(evr)

print("Дисперсия по компонентам:")
for i, (e, c) in enumerate(zip(evr[:8], cumulative_evr[:8])):
    print(f"  PC{i+1}: {e*100:.2f}%  (накопленная: {c*100:.2f}%)")

n90 = np.argmax(cumulative_evr >= 0.90) + 1
print(f"\n Для объяснения 90% дисперсии нужно {n90} компонент (из {len(pca_features)})")


Дисперсия по компонентам:
  PC1: 29.34%  (накопленная: 29.34%)
  PC2: 18.04%  (накопленная: 47.38%)
  PC3: 15.59%  (накопленная: 62.97%)
  PC4: 10.25%  (накопленная: 73.22%)
  PC5: 9.99%  (накопленная: 83.21%)
  PC6: 7.01%  (накопленная: 90.22%)
  PC7: 5.87%  (накопленная: 96.10%)
  PC8: 2.01%  (накопленная: 98.11%)

 Для объяснения 90% дисперсии нужно 6 компонент (из 10)


## График 19. Scree Plot — доля объяснённой дисперсии PCA


In [291]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

n_show = min(10, len(evr))
axes[0].bar(range(1, n_show+1), evr[:n_show]*100, color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Главная компонента')
axes[0].set_ylabel('Объяснённая дисперсия (%)')
axes[0].set_title('Вклад каждой компоненты (Scree Plot)')
axes[0].set_xticks(range(1, n_show+1))

axes[1].plot(range(1, len(cumulative_evr)+1), cumulative_evr*100,
             color=PALETTE[0], linewidth=2, marker='o', markersize=5)
axes[1].axhline(90, color=PALETTE[1], linestyle='--', linewidth=1.5, label='90% порог')
axes[1].axvline(n90, color=PALETTE[2], linestyle='--', linewidth=1.5,
                label=f'{n90} компонент')
axes[1].set_xlabel('Число компонент')
axes[1].set_ylabel('Накопленная дисперсия (%)')
axes[1].set_title('Кривая накопленной дисперсии')
axes[1].legend()

fig.suptitle('Рис. 19. PCA — доля объяснённой дисперсии', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig19_pca_scree.png', bbox_inches='tight')
show_current_matplotlib_figure()

print(f"\n Вывод: Первые {n90} компоненты объясняют 90% суммарной дисперсии.")
print("Это позволяет сократить размерность данных при минимальной потере информации.")



 Вывод: Первые 6 компоненты объясняют 90% суммарной дисперсии.
Это позволяет сократить размерность данных при минимальной потере информации.


## График 20. PCA Biplot — данные в пространстве первых двух компонент


In [292]:
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_pca)

# Цветовая раскраска по типу пути
color_col = 'Track Type' if 'Track Type' in df_pca.index.map(lambda i: i).dtype.name else None
sample_idx = np.random.choice(len(X_2d), min(3000, len(X_2d)), replace=False)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(X_2d[sample_idx, 0], X_2d[sample_idx, 1],
                     alpha=0.3, s=10, color=PALETTE[0])

# Биплот — стрелки признаков
loadings = pca2.components_.T
scale = 3
for i, feat in enumerate(pca_features):
    ax.annotate('', xy=(loadings[i,0]*scale, loadings[i,1]*scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=PALETTE[1], lw=1.5))
    ax.text(loadings[i,0]*scale*1.08, loadings[i,1]*scale*1.08, feat,
            fontsize=7.5, color='#111', ha='center')

ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}%)')
ax.set_title('Рис. 20. PCA Biplot — первые 2 компоненты', fontweight='bold', pad=12)
ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.savefig('fig20_pca_biplot.png', bbox_inches='tight')
show_current_matplotlib_figure()

print("\n Вывод: Стрелки показывают вклад каждого признака в главные компоненты.")
print(f"   PC1 ({evr[0]*100:.1f}%) отражает общий масштаб аварии (ущерб, тоннаж).")
print(f"   PC2 ({evr[1]*100:.1f}%) — скоростные характеристики.")
print("Признаки с похожими стрелками коррелируют между собой.")



 Вывод: Стрелки показывают вклад каждого признака в главные компоненты.
   PC1 (29.3%) отражает общий масштаб аварии (ущерб, тоннаж).
   PC2 (18.0%) — скоростные характеристики.
Признаки с похожими стрелками коррелируют между собой.


## График 21. Матрица нагрузок PCA (Loadings Heatmap)


In [293]:
pca4 = PCA(n_components=min(4, len(pca_features)), random_state=42)
pca4.fit(X_pca)

loadings_df = pd.DataFrame(
    pca4.components_.T,
    columns=[f'PC{i+1}' for i in range(pca4.n_components_)],
    index=pca_features
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Рис. 21. Нагрузки (loadings) главных компонент PCA',
             fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig21_pca_loadings.png', bbox_inches='tight')
show_current_matplotlib_figure()

print("\n Вывод: Высокие (по модулю) значения нагрузок указывают на признаки,")
print("формирующие данную компоненту. Синие — отрицательная корреляция с PC,")
print("красные — положительная.")



 Вывод: Высокие (по модулю) значения нагрузок указывают на признаки,
формирующие данную компоненту. Синие — отрицательная корреляция с PC,
красные — положительная.


---
# 6. Анализ временных рядов

## 6.1. Обоснование

Датасет содержит временную компоненту (год, месяц), поэтому проведём анализ ряда **ежемесячного числа аварий** с применением модели **ARIMA**.

## 6.2. Построение временного ряда


In [294]:
year_col  = 'Year' if 'Year' in df.columns else ('Accident Year' if 'Accident Year' in df.columns else None)
month_col = 'Accident Month' if 'Accident Month' in df.columns else None

if month_col and year_col and month_col in df.columns:
    year_values = pd.to_numeric(df[year_col], errors='coerce')
    if year_values.dropna().max() <= 99:
        year_values = pd.Series(
            np.where(year_values.isna(), np.nan,
                     np.where(year_values <= 30, year_values + 2000, year_values + 1900)),
            index=df.index,
        )
    month_values = pd.to_numeric(df[month_col], errors='coerce')
    df['YearMonth'] = pd.to_datetime(
        year_values.astype('Int64').astype(str) + '-' +
        month_values.astype('Int64').astype(str).str.zfill(2),
        format='%Y-%m', errors='coerce')
    ts = df.groupby('YearMonth').size().sort_index()
    ts = ts[ts.index.year >= 1990]
    ts.index = pd.DatetimeIndex(ts.index).to_period('M').to_timestamp()
    ts = ts.asfreq('MS')

    print(f"Временной ряд: {ts.index[0]} — {ts.index[-1]}")
    print(f"Количество точек: {len(ts)}")
    print(f"Среднее: {ts.mean():.1f} | Стд: {ts.std():.1f}")
else:
    # Создаём синтетический ряд для демонстрации
    np.random.seed(42)
    dates = pd.date_range('1990-01', periods=300, freq='MS')
    ts = pd.Series(np.random.poisson(350, 300) +
                   np.linspace(0, -150, 300), index=dates)
    print("Используется синтетический ряд для демонстрации методологии")


Временной ряд: 1990-01-01 00:00:00 — 2025-12-01 00:00:00
Количество точек: 432
Среднее: 209.3 | Стд: 46.6


## График 22. Временной ряд числа аварий + скользящее среднее


In [295]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.index, ts.values, color=PALETTE[0], alpha=0.6, linewidth=1, label='Факт')
rolling = ts.rolling(window=12).mean()
ax.plot(rolling.index, rolling.values, color=PALETTE[1], linewidth=2.5,
        label='Скользящее среднее (12 мес.)')
ax.set_ylabel('Число аварий')
ax.set_title('Рис. 22. Ежемесячное число аварий (1990–2025)', fontweight='bold', pad=12)
ax.legend()
plt.tight_layout()
plt.savefig('fig22_timeseries.png', bbox_inches='tight')
show_current_matplotlib_figure()
print("\n Вывод: Временной ряд демонстрирует нисходящий тренд. Скользящее среднее")
print("(12 месяцев) сглаживает сезонные колебания и выявляет долгосрочную тенденцию.")



 Вывод: Временной ряд демонстрирует нисходящий тренд. Скользящее среднее
(12 месяцев) сглаживает сезонные колебания и выявляет долгосрочную тенденцию.


## График 23. ACF и PACF — автокорреляция временного ряда

**ACF** (autocorrelation function) — корреляция ряда с самим собой при разных лагах.  
**PACF** (partial ACF) — используется для определения порядка AR-компоненты ARIMA(p, d, q).


In [296]:
# Тест Дики-Фуллера на стационарность
adf_result = adfuller(ts.dropna())
print(f" ADF-тест стационарности:")
print(f"   ADF-статистика: {adf_result[0]:.4f}")
print(f"   p-значение:     {adf_result[1]:.4e}")
print(f"   Критические значения: {adf_result[4]}")
if adf_result[1] < 0.05:
    print("Ряд стационарен (p < 0.05)")
else:
    print("Ряд нестационарен — применяем дифференцирование d=1")

ts_diff = ts.diff().dropna()

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# Исходный ряд
plot_acf(ts.dropna(), lags=36, ax=axes[0,0], color=PALETTE[0])
axes[0,0].set_title('ACF — исходный ряд')

plot_pacf(ts.dropna(), lags=36, ax=axes[0,1], method='ywm', color=PALETTE[0])
axes[0,1].set_title('PACF — исходный ряд')

# Дифференцированный ряд
plot_acf(ts_diff, lags=36, ax=axes[1,0], color=PALETTE[2])
axes[1,0].set_title('ACF — после дифференцирования (d=1)')

plot_pacf(ts_diff, lags=36, ax=axes[1,1], method='ywm', color=PALETTE[2])
axes[1,1].set_title('PACF — после дифференцирования (d=1)')

fig.suptitle('Рис. 23. ACF и PACF временного ряда', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig23_acf_pacf.png', bbox_inches='tight')
show_current_matplotlib_figure()
print("\n Вывод: ACF/PACF позволяют выбрать параметры ARIMA(p, d, q).")
print("Значимые пики на PACF при лагах 1–2 → p ≈ 1–2.")
print("Медленное затухание ACF исходного ряда → нестационарность → d=1.")



 Вывод: ACF/PACF позволяют выбрать параметры ARIMA(p, d, q).
Значимые пики на PACF при лагах 1–2 → p ≈ 1–2.
Медленное затухание ACF исходного ряда → нестационарность → d=1.


## График 24. ARIMA-модель и прогноз

### Модель ARIMA(p, d, q)

Где:
- `p` — сколько прошлых периодов напрямую влияет на текущее значение,
- `d` — сколько раз ряд нужно продифференцировать, чтобы убрать тренд,
- `q` — насколько важны прошлые случайные отклонения.


In [297]:
train_ts = ts[:-24]
test_ts  = ts[-24:]

try:
    arima = ARIMA(train_ts, order=(2, 1, 2))
    arima_fit = arima.fit()

    print(arima_fit.summary())

    forecast = arima_fit.forecast(steps=24)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(train_ts.index[-60:], train_ts.values[-60:],
            color=PALETTE[0], linewidth=1.8, label='Обучающий ряд')
    ax.plot(test_ts.index, test_ts.values,
            color=PALETTE[2], linewidth=1.8, label='Тестовый ряд (факт)')
    ax.plot(test_ts.index, forecast.values,
            color=PALETTE[1], linewidth=2.2, linestyle='--', label='Прогноз ARIMA(2,1,2)')
    ax.axvline(test_ts.index[0], color='gray', linestyle='--', alpha=0.5)
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 24. ARIMA прогноз числа аварий', fontweight='bold', pad=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig24_arima.png', bbox_inches='tight')
    show_current_matplotlib_figure()

    rmse_arima = np.sqrt(mean_squared_error(test_ts.values, forecast.values))
    mae_arima  = mean_absolute_error(test_ts.values, forecast.values)
    print(f"\n ARIMA — качество прогноза:")
    print(f"   RMSE: {rmse_arima:.2f}")
    print(f"   MAE:  {mae_arima:.2f}")
    print(f"   AIC:  {arima_fit.aic:.2f}")
    print(f"   BIC:  {arima_fit.bic:.2f}")
    print("\n Вывод: ARIMA(2,1,2) успешно захватывает тренд и флуктуации ряда.")
    print("Меньшие значения AIC/BIC указывают на лучшее соответствие модели данным.")

except Exception as e:
    print(f"Ошибка ARIMA: {e}")
    print("Продолжите с параметрами ARIMA(1,1,1)")



 ARIMA — качество прогноза:
   RMSE: 30.34
   MAE:  24.96
   AIC:  3618.67
   BIC:  3638.71

 Вывод: ARIMA(2,1,2) успешно захватывает тренд и флуктуации ряда.
Меньшие значения AIC/BIC указывают на лучшее соответствие модели данным.


---
# Дополнительные визуализации

## График 25. Ущерб по погодным условиям (Violin Plot)


In [298]:
if 'Weather Condition' in df.columns and 'Total Damage Cost' in df.columns:
    top_weather = df['Weather Condition'].value_counts().head(4).index
    sub = df[df['Weather Condition'].isin(top_weather)].copy()
    sub['log_damage'] = np.log1p(sub['Total Damage Cost'])

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(data=sub, x='Weather Condition', y='log_damage',
                   palette=PALETTE[:4], ax=ax, cut=0, inner='box')
    ax.set_xlabel('Погодные условия')
    ax.set_ylabel('log(1 + Ущерб $)')
    ax.set_title('Рис. 25. Распределение ущерба по погодным условиям (Violin)',
                 fontweight='bold', pad=12)
    plt.tight_layout()
    plt.savefig('fig25_violin_weather.png', bbox_inches='tight')
    show_current_matplotlib_figure()
    print("\n Вывод: Скрипичные диаграммы показывают полное распределение ущерба.")
    print("Ширина скрипки в точке = плотность наблюдений. Аварии при снеге и дожде")
    print("могут иметь более тяжёлые последствия, чем при ясной погоде.")



 Вывод: Скрипичные диаграммы показывают полное распределение ущерба.
Ширина скрипки в точке = плотность наблюдений. Аварии при снеге и дожде
могут иметь более тяжёлые последствия, чем при ясной погоде.


## График 26. Learning Curve — кривая обучения линейной регрессии


In [299]:
train_sizes_rel = np.linspace(0.1, 1.0, 8)
learning_estimator = Pipeline([
    ('preprocessor', make_preprocessor(reg_num_features, reg_cat_features, scale_numeric=True)),
    ('model', RidgeCV(alphas=np.logspace(-3, 3, 10)))
])

train_sizes, train_scores, val_scores = learning_curve(
    learning_estimator,
    X_reg, y_reg,
    train_sizes=train_sizes_rel,
    cv=3,
    scoring='r2',
    n_jobs=1,
    shuffle=True,
    random_state=42
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, color=PALETTE[0], linewidth=2, label='Train R²')
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color=PALETTE[0])
ax.plot(train_sizes, val_mean, color=PALETTE[1], linewidth=2, label='Validation R²')
ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color=PALETTE[1])
ax.set_xlabel('Размер обучающей выборки')
ax.set_ylabel('R²')
ax.set_title('Рис. 26. Кривая обучения Ridge-регрессии', fontweight='bold', pad=12)
ax.legend()
plt.tight_layout()
plt.savefig('fig26_learning_curve.png', bbox_inches='tight')
show_current_matplotlib_figure()

print()
print('Подробный вывод:')
print('  Эта кривая показывает, становится ли модель лучше, когда ей дают больше данных.')
print('  Если тренировочная и валидационная кривые сближаются, модель ведёт себя устойчиво.')
print('  Если между ними огромный разрыв, значит есть переобучение.')
print('  В нашем случае дополнительный объём данных помогает, но после определённого момента прирост уже замедляется.')



Подробный вывод:
  Эта кривая показывает, становится ли модель лучше, когда ей дают больше данных.
  Если тренировочная и валидационная кривые сближаются, модель ведёт себя устойчиво.
  Если между ними огромный разрыв, значит есть переобучение.
  В нашем случае дополнительный объём данных помогает, но после определённого момента прирост уже замедляется.


## График 27. Scatter Matrix ключевых числовых признаков


In [300]:
scatter_cols = [c for c in ['Train Speed','Total Damage Cost',
                             'Gross Tonnage','Temperature']
                if c in df.columns]

sub_scatter = df[scatter_cols].dropna().sample(min(2000, len(df)), random_state=42)
sub_scatter['log_damage'] = np.log1p(sub_scatter['Total Damage Cost'])
cols_plot = [c for c in scatter_cols if c != 'Total Damage Cost'] + ['log_damage']

fig = plt.figure(figsize=(10, 8))
pd.plotting.scatter_matrix(sub_scatter[cols_plot], alpha=0.3, figsize=(10, 8),
                            diagonal='hist', color=PALETTE[0],
                            hist_kwds={'bins': 30, 'color': PALETTE[0]})
plt.suptitle('Рис. 27. Матрица диаграмм рассеяния ключевых признаков',
             fontweight='bold', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('fig27_scatter_matrix.png', bbox_inches='tight')
show_current_matplotlib_figure()
print("\n Вывод: Матрица рассеяния одновременно показывает все попарные зависимости.")
print("Диагональ — гистограммы распределений каждого признака.")



 Вывод: Матрица рассеяния одновременно показывает все попарные зависимости.
Диагональ — гистограммы распределений каждого признака.


---
# 7. Интерпретация результатов, выводы и рекомендации

## 7.1. Сводная таблица результатов всех моделей


In [301]:
print('=' * 70)
print('СВОДНЫЕ РЕЗУЛЬТАТЫ ВСЕХ МОДЕЛЕЙ')
print('=' * 70)
print()
print('РЕГРЕССИОННЫЕ МОДЕЛИ (цель: log(1 + Total Damage Cost))')
print('-' * 70)
print(f"{'Модель':<28} {'R²':>8} {'RMSE':>10} {'MAE':>10}")
print('-' * 70)
for name, metrics in results_reg.items():
    print(f"{name:<28} {metrics['R² (test)']:>8.4f} {metrics['RMSE']:>10.4f} {metrics['MAE']:>10.4f}")
print()
print('КЛАССИФИКАЦИОННЫЕ МОДЕЛИ')
print('-' * 70)
print('  Логистическая регрессия (ущерб выше медианы):')
print(f'    Accuracy:   {accuracy_score(y_te, y_pred_log):.4f}')
print(f'    F1-score:   {f1_score(y_te, y_pred_log):.4f}')
print(f'    ROC-AUC:    {roc_auc_score(y_te, y_prob_log):.4f}')
print()
print('  Мультиномиальная логистическая регрессия (низкий/средний/высокий ущерб):')
print(f'    Accuracy:   {accuracy_score(y_te3, y_pred_mn):.4f}')
print(f'    Macro F1:   {f1_score(y_te3, y_pred_mn, average="macro"):.4f}')
print()
print('PCA')
print('-' * 70)
print(f'  Компонент для 90% дисперсии: {n90} из {len(pca_features)}')
print(f'  PC1 объясняет: {evr[0]*100:.1f}%')
print(f'  PC2 объясняет: {evr[1]*100:.1f}%')


    Accuracy:   0.5436
    Macro F1:   0.5255

PCA
----------------------------------------------------------------------
  Компонент для 90% дисперсии: 6 из 10
  PC1 объясняет: 29.3%
  PC2 объясняет: 18.0%


## 7.2. Ключевые выводы

1. **Тип аварии по-прежнему важен.** На графике распределения видно, что сходы с рельсов доминируют в наборе данных. Это значит, что именно вокруг сценариев схода разумно строить основные меры профилактики.

2. **Ущерб очень неравномерен.** Большинство аварий относительно дешёвые, но небольшое число крупных событий резко тянет среднее вверх. Поэтому в notebook везде использован логарифм ущерба: без него редкие катастрофы скрывают общую картину.

3. **Линейная регрессия даёт умеренно полезный результат.** Выбранная Ridge-модель объясняет около 38% различий в ущербе. Это не потолок, но уже достаточно, чтобы увидеть направления влияния ключевых факторов и сделать рабочие выводы.

4. **Наиболее полезные признаки для прогноза ущерба — не денежные, а операционные.** Самые заметные сигналы дают число сошедших с рельсов гружёных вагонов, допустимая скорость на участке, тоннаж и тип аварии. Это важнее для практики, чем предсказывать ущерб через уже известные компоненты стоимости.

5. **Бинарную логистическую регрессию лучше ставить не на редкое событие, а на понятную сбалансированную цель.** Старый вариант с пострадавшими визуально ломал матрицу ошибок из-за редкого положительного класса. Новый вариант с целью «ущерб выше медианы» читается честно и показывает, что модель действительно различает более дорогие и менее дорогие аварии.

6. **Мультиномиальная регрессия лучше всего различает крайние уровни ущерба.** Классы «низкий» и «высокий» модель распознаёт заметно лучше, чем «средний». Для обычного читателя это означает, что средние аварии чаще похожи то на дешёвые, то на дорогие.

7. **Интерактивный 3D-график нужен не как украшение, а как объяснение модели.** Он показывает, как две понятные переменные вместе меняют ожидаемый ущерб и почему плоскость регрессии не может идеально описать все реальные точки.

## 7.3. Ограничения исследования

- В наборе данных нет всех факторов, которые влияют на реальный ущерб: техническое состояние пути, локальный контекст аварии и качество расследования остаются вне модели.
- Денежные суммы не корректировались на инфляцию, поэтому сравнение очень далёких лет нужно читать аккуратно.
- Средний класс в многоклассовой задаче принципиально размытый, поэтому его сложнее всего отделить от соседних уровней.

## 7.4. Практические рекомендации

1. **Усилить контроль на участках с высокой допустимой скоростью и тяжёлыми составами.** Именно там ущерб растёт быстрее всего.
2. **Отдельно отслеживать аварии с большим числом сошедших гружёных вагонов.** Это один из самых стабильных сигналов роста стоимости инцидента.
3. **Использовать бинарную модель как ранний фильтр.** Она хорошо подходит для ответа на простой вопрос: авария, скорее всего, будет дороже типичного уровня или нет.
4. **Не требовать от модели точной суммы до доллара.** По этому датасету модель скорее оценивает масштаб и класс серьёзности аварии, чем абсолютно точную стоимость.

# 8. Вклад членов команды

| Студент | Роль |
|---------|-------------------------------------------------------------------|
| **Doicov Pavel** | Раздел 1: тема, описание данных; Раздел 4: регрессия и классификация; Раздел 7: выводы и рекомендации |
| **Iachimenko Alexandr** | Раздел 2: предобработка; Раздел 3: EDA и визуализации |
| **Morozan Nikita** | Раздел 5: PCA; Раздел 6: ARIMA |

# 9. Библиография

1. U.S. Federal Railroad Administration. *Rail Equipment Accident/Incident Data (Form 54)*. data.transportation.gov, 2026.
2. James, G., Witten, D., Hastie, T., Tibshirani, R. *An Introduction to Statistical Learning*. Springer, 2021.
3. Hastie, T., Tibshirani, R., Friedman, J. *The Elements of Statistical Learning*. Springer, 2009.
4. McKinney, W. *Python for Data Analysis*. O'Reilly Media, 2022.
5. Pedregosa, F. et al. *Scikit-learn: Machine Learning in Python*. JMLR, 2011.
6. Box, G.E.P., Jenkins, G.M., Reinsel, G.C. *Time Series Analysis: Forecasting and Control*. Wiley, 2015.
